# 01 — QuPath ingest and geometry QC

This notebook is an output-first view of the production `ingest` and `geometry` stages. It contains no ingestion, rasterization, duplicate-selection, or QC algorithms. The immutable QuPath cohort manifest supplies donor/image identity, panel mappings, geometry, pixel calibration, and source fingerprints; production APIs own all writes.

Run the status and inspection cells first. Set `RUN_STAGES = True` only when you intend to create the content-addressed stage outputs.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from phenocycler.artifacts import StageManifest
from phenocycler.config import load_config
from phenocycler.pipeline import RunContext, resolve_run_context, run_stage, status

CONFIG_PATH = None  # optionally set a repository-relative Path to another config.ini
cfg = load_config(CONFIG_PATH)
context: RunContext = resolve_run_context(cfg)
print(f"run_id={context.run_id}  donors={len(context.donors)}  root={context.run_root}")

In [ ]:
# Evidence-backed and non-mutating: validates manifests, exact donor sets, config, code and cached file stats.
status_code = status(context)
print(f"status return code: {status_code}")

## Source contract

There is exactly one manifest record per donor/image. Images sharing a panel ID must share an exact marker-to-channel mapping. Fast validation does not reread unchanged qptiffs; use content validation outside routine status when a full audit is required.

In [ ]:
cohort_rows = [
    {
        "donor_id": image.donor_id,
        "image_id": image.image_id,
        "panel_id": image.panel_id,
        "channels": len(image.channel_map),
        "pixel_size_um_x": image.pixel_size_um_x,
        "pixel_size_um_y": image.pixel_size_um_y,
        "segmentation_version": image.segmentation_version,
        "objects": image.cell_geometry.feature_count,
        "geometry_content": image.content_id[:12],
    }
    for image in context.cohort.images
]
display(pd.DataFrame(cohort_rows))
display(pd.DataFrame([
    {"panel_id": panel.panel_id, "channels": len(panel.channels), "content_id": panel.content_id[:12]}
    for panel in context.cohort.panels
]))

## Optional production execution

`run_stage` either validates an existing immutable stage or runs the production implementation and writes its content-derived manifest. It refuses partial output directories.

In [ ]:
RUN_STAGES = False

if RUN_STAGES:
    for stage_name in ("ingest", "geometry"):
        run_stage(context, stage_name)
else:
    print("Inspection only. Set RUN_STAGES=True to run ingest and geometry through production APIs.")

In [ ]:
manifest_rows = []
for stage_name in ("ingest", "geometry"):
    path = context.stage_manifest_path(stage_name)
    if path.exists():
        manifest = StageManifest.read_json(path)
        manifest_rows.append({
            "stage": stage_name,
            "method_version": manifest.method_version,
            "donors": len(manifest.completed_donors),
            "rows": manifest.output.total_rows,
            "schema": manifest.output.schema_sha256[:12],
            "objects": manifest.output.object_id_sha256[:12],
            "content": manifest.content_id[:12],
        })
display(pd.DataFrame(manifest_rows))

## Inspect one donor

Geometry QC preserves the complete ingest table and emits three separate decisions: `analysis_eligible`, `estimation_eligible`, and `spillover_context_eligible`. No marker intensity or REDSEA result participates in these flags.

In [ ]:
DONOR = context.donors[0]
required = (context.stage_manifest_path("ingest"), context.stage_manifest_path("geometry"))
if all(path.exists() for path in required):
    cell_path = context.config.cells_dir / f"donor_id={DONOR}" / "*.parquet"
    geometry_files = sorted((context.config.geometry_qc_dir / f"donor_id={DONOR}").glob("*.parquet"))
    if len(geometry_files) != 1:
        raise RuntimeError(f"expected one geometry partition for {DONOR}, found {len(geometry_files)}")
    cell_count = duckdb.sql("SELECT count(*) FROM read_parquet(?)", params=[cell_path.as_posix()]).fetchone()[0]
    geometry_columns = [
        "object_id", "analysis_eligible", "estimation_eligible",
        "spillover_context_eligible", "analysis_reasons", "analysis_reason",
        "raster_area_ratio", "nucleus_cell_area_ratio",
    ]
    geometry = pd.read_parquet(geometry_files[0], columns=geometry_columns)
    print(f"donor {DONOR}: cells={cell_count:,}, geometry decisions={len(geometry):,}")
    eligibility = [
        "analysis_eligible", "estimation_eligible", "spillover_context_eligible"
    ]
    display(geometry[eligibility].mean().rename("eligible_fraction").to_frame())
    display(
        geometry.groupby("analysis_reason", dropna=False).size()
        .rename("cells").sort_values(ascending=False).to_frame().head(15)
    )
    preview_columns = [
        column for column in (
            "object_id", "analysis_eligible", "estimation_eligible",
            "spillover_context_eligible", "analysis_reasons",
            "raster_area_ratio", "nucleus_cell_area_ratio"
        ) if column in geometry
    ]
    display(geometry.loc[:, preview_columns].head())
else:
    print("Ingest/geometry artifacts are not complete yet; inspect status or run the optional stage cell.")

## Cohort geometry decision plots

The first plot localizes exclusions by donor and role. The second shows why cells were excluded from analysis. These are not automatic release gates: every major exclusion reason needs a random image-linked audit for debris/split-mask false positives and missed/merged-cell false negatives. The selected-donor distributions show where geometry decisions fall, but their shapes must not be tuned to make donors look alike.

In [ ]:
if context.stage_manifest_path("geometry").exists():
    geometry_glob = (context.config.geometry_qc_dir / "donor_id=*" / "*.parquet").as_posix()
    connection = duckdb.connect()
    connection.execute(f"SET threads={int(context.config.duckdb_threads)}")
    role_summary = connection.execute(
        """
        SELECT CAST(donor_id AS VARCHAR) AS donor_id, count(*) AS cells,
               avg(CAST(analysis_eligible AS DOUBLE)) AS analysis,
               avg(CAST(estimation_eligible AS DOUBLE)) AS estimation,
               avg(CAST(spillover_context_eligible AS DOUBLE)) AS spillover_context
        FROM read_parquet(?) GROUP BY 1 ORDER BY 1
        """, [geometry_glob],
    ).fetchdf().set_index("donor_id")
    reason_summary = connection.execute(
        """
        SELECT CAST(donor_id AS VARCHAR) AS donor_id, analysis_reason, count(*) AS cells
        FROM read_parquet(?) GROUP BY 1, 2
        """, [geometry_glob],
    ).fetchdf()

    fig, axes = plt.subplots(1, 2, figsize=(18, max(6, 0.34 * len(role_summary))), constrained_layout=True)
    role_summary[["analysis", "estimation", "spillover_context"]].plot.barh(
        ax=axes[0], color=["#4c78a8", "#f2cf5b", "#59a14f"]
    )
    axes[0].set_xlim(0, 1)
    axes[0].set_xlabel("Eligible fraction")
    axes[0].set_ylabel("Donor")
    axes[0].set_title("Geometry eligibility by role")
    axes[0].grid(axis="x", alpha=0.2)

    top_reasons = (
        reason_summary.groupby("analysis_reason", dropna=False)["cells"].sum()
        .sort_values(ascending=False).head(12).index
    )
    reason_matrix = (
        reason_summary.loc[reason_summary["analysis_reason"].isin(top_reasons)]
        .pivot(index="donor_id", columns="analysis_reason", values="cells")
        .fillna(0).reindex(index=role_summary.index, columns=top_reasons, fill_value=0)
    )
    reason_fraction = reason_matrix.div(role_summary["cells"], axis=0)
    image = axes[1].imshow(reason_fraction.to_numpy(), aspect="auto", cmap="magma")
    axes[1].set_xticks(np.arange(len(reason_fraction.columns)), labels=reason_fraction.columns, rotation=55, ha="right")
    axes[1].set_yticks(np.arange(len(reason_fraction.index)), labels=reason_fraction.index)
    axes[1].set_xlabel("Analysis decision reason")
    axes[1].set_ylabel("Donor")
    axes[1].set_title("Cell fraction by geometry reason")
    fig.colorbar(image, ax=axes[1], label="Fraction of all donor objects")
    plt.show()

    if 'geometry' in locals() and not geometry.empty:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
        for eligible, color, label in ((True, '#4c78a8', 'analysis eligible'), (False, '#e45756', 'analysis excluded')):
            selected = geometry.loc[geometry['analysis_eligible'].astype(bool).eq(eligible)]
            for ax, column, xlabel in (
                (axes[0], 'raster_area_ratio', 'Raster / source cell area'),
                (axes[1], 'nucleus_cell_area_ratio', 'Nucleus / cell area'),
            ):
                values = pd.to_numeric(selected[column], errors='coerce').dropna()
                if len(values):
                    low, high = values.quantile([0.005, 0.995])
                    ax.hist(values.clip(low, high), bins=80, density=True, histtype='step', linewidth=1.5, color=color, label=label)
                ax.set_xlabel(xlabel)
                ax.set_ylabel('Density')
                ax.grid(alpha=0.2)
        axes[0].legend()
        fig.suptitle(f'Donor {DONOR} geometry distributions (tails clipped for display)')
        plt.show()
else:
    print("Geometry manifest is absent; cohort plots are unavailable.")

## Handoff

Proceed to notebook 02 only when both manifests are `CURRENT`. REDSEA consumes the immutable ingest and geometry universes; it no longer serves as a prerequisite for geometry QC.